In [1]:
# ============================================
# Startup Cell: Mount Drive + Prepare Data
# ============================================

from google.colab import drive
drive.mount("/content/drive")

import os
import shutil

BASE_DRIVE_DIR = "/content/drive/MyDrive/DIP_Project"
CONTENT_DIR = "/content"

# -------------------------------------------------
# Input metadata CSVs (from preprocessing stage)
# -------------------------------------------------
INPUT_FILES = [
    "imagenet_1k_256_preprocessed_metadata.csv",
    "ms_coco_2017_preprocessed_metadata.csv",
    "diffusiondb_preprocessed_metadata.csv",
    "sdxl_generated_10k_preprocessed_metadata.csv"
]

# -------------------------------------------------
# Copy metadata CSVs to /content
# -------------------------------------------------
print("Copying metadata CSV files...")

for filename in INPUT_FILES:
    src = os.path.join(BASE_DRIVE_DIR, filename)
    dst = os.path.join(CONTENT_DIR, filename)

    if not os.path.exists(src):
        raise FileNotFoundError(f"Missing source file: {src}")

    shutil.copy(src, dst)
    print(f"Copied: {filename}")

print("\nMetadata CSVs copied.")

# -------------------------------------------------
# Verification
# -------------------------------------------------
print("\nVerification:")
all_ok = True

for filename in INPUT_FILES:
    path = os.path.join(CONTENT_DIR, filename)
    exists = os.path.exists(path)
    print(f"{filename:<50} {'OK' if exists else 'MISSING'}")
    all_ok = all_ok and exists

print("\nAll files present." if all_ok else "\nSome files are missing.")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copying metadata CSV files...
Copied: imagenet_1k_256_preprocessed_metadata.csv
Copied: ms_coco_2017_preprocessed_metadata.csv
Copied: diffusiondb_preprocessed_metadata.csv
Copied: sdxl_generated_10k_preprocessed_metadata.csv

Metadata CSVs copied.

Verification:
imagenet_1k_256_preprocessed_metadata.csv          OK
ms_coco_2017_preprocessed_metadata.csv             OK
diffusiondb_preprocessed_metadata.csv              OK
sdxl_generated_10k_preprocessed_metadata.csv       OK

All files present.


In [2]:
# ============================================
# Cell 0: Notebook Overview
# ============================================
# Purpose:
#   This notebook combines 4 preprocessed dataset metadata CSV files
#   and creates balanced train / validation / test split CSV files.
#
# Inputs:
#   The notebook expects these 4 preprocessed metadata CSV files:
#     /content/preprocessed_csvs/imagenet_1k_256_preprocessed_metadata.csv
#     /content/preprocessed_csvs/ms_coco_2017_preprocessed_metadata.csv
#     /content/preprocessed_csvs/diffusiondb_preprocessed_metadata.csv
#     /content/preprocessed_csvs/sdxl_generated_10k_preprocessed_metadata.csv
#
# Assumptions:
#   - Each input CSV contains a 'filename' column.
#   - Each input CSV contains 3000 rows.
#   - The datasets correspond to:
#       ImageNet_1K_256       -> real
#       MS_COCO_2017          -> real
#       DiffusionDB           -> ai
#       SDXL_Generated_10K    -> ai
#   - This notebook does NOT move, copy, unzip, or preprocess images.
#   - This notebook only creates metadata CSV files.
#
# What the notebook does:
#   Cell 1:
#     Import libraries and define input/output paths.
#
#   Cell 2:
#     Define dataset configuration, including CSV filenames and class labels.
#
#   Cell 3:
#     Load the 4 input CSV files, add:
#       - source_dataset
#       - class_label
#       - image_path
#     Then combine them into one master dataframe and save:
#       combined_metadata.csv
#
#   Cell 4:
#     For each dataset independently:
#       1. Shuffle rows
#       2. Split into exact counts:
#            train      = 2100
#            validation =  450
#            test       =  450
#     Then:
#       - combine all train splits
#       - combine all validation splits
#       - combine all test splits
#       - shuffle each final subset independently
#     Then save:
#       train_metadata.csv
#       validation_metadata.csv
#       test_metadata.csv
#
#   Cell 5:
#     Perform an optional validation that the 4 output CSV files exist
#     and report their shapes.
#
# Outputs:
#   /content/combined_metadata.csv
#   /content/train_metadata.csv
#   /content/validation_metadata.csv
#   /content/test_metadata.csv
#
# Expected final sizes:
#   combined_metadata.csv     -> 12000 rows
#   train_metadata.csv        ->  8400 rows
#   validation_metadata.csv   ->  1800 rows
#   test_metadata.csv         ->  1800 rows
#
# Notes:
#   - The split is balanced by source dataset and by class label.
#   - The split is reproducible because a fixed random seed is used.
#   - Images remain in their existing storage location; the CSV files
#     act as the split definitions for later notebooks.
# ============================================

print("Notebook overview loaded.")


Notebook overview loaded.


In [3]:
# =========================
# Cell 1: Imports and base paths
# =========================
from pathlib import Path
import pandas as pd

BASE_DIR = Path("/content")

# Directory containing the 4 preprocessed CSV files
INPUT_CSV_DIR = BASE_DIR

# Output directory for combined/split metadata
OUTPUT_DIR = BASE_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42

print("Looking for CSVs in:", INPUT_CSV_DIR)
print("Files found:")
for f in INPUT_CSV_DIR.glob("*.csv"):
    print(" -", f.name)



Looking for CSVs in: /content
Files found:
 - imagenet_1k_256_preprocessed_metadata.csv
 - validation_metadata.csv
 - combined_metadata.csv
 - ms_coco_2017_preprocessed_metadata.csv
 - sdxl_generated_10k_preprocessed_metadata.csv
 - test_metadata.csv
 - train_metadata.csv
 - diffusiondb_preprocessed_metadata.csv


In [4]:
# =========================
# Cell 2: Dataset configuration
# =========================
DATASET_CONFIG = {
    "ImageNet_1K_256": {
        "csv": "imagenet_1k_256_preprocessed_metadata.csv",
        "class_label": "real",
    },
    "MS_COCO_2017": {
        "csv": "ms_coco_2017_preprocessed_metadata.csv",
        "class_label": "real",
    },
    "DiffusionDB": {
        "csv": "diffusiondb_preprocessed_metadata.csv",
        "class_label": "ai",
    },
    "SDXL_Generated_10K": {
        "csv": "sdxl_generated_10k_preprocessed_metadata.csv",
        "class_label": "ai",
    },
}



In [5]:
# =========================
# Cell 3: Load and combine metadata
# =========================
dfs = []

BASE_COLUMNS = ["filename"]

for dataset_name, cfg in DATASET_CONFIG.items():
    csv_path = INPUT_CSV_DIR / cfg["csv"]

    if not csv_path.exists():
        raise FileNotFoundError(f"Missing input CSV: {csv_path}")

    df = pd.read_csv(csv_path)

    if "filename" not in df.columns:
        raise ValueError(f"'filename' column not found in {csv_path}")

    # Keep only required columns
    df = df[BASE_COLUMNS].copy()

    # Add standardized fields
    df["source_dataset"] = dataset_name
    df["class_label"] = cfg["class_label"]
    df["image_path"] = df["filename"].apply(lambda x: f"{dataset_name}/images/{x}")

    dfs.append(df)

combined_df = pd.concat(dfs, ignore_index=True)

print("Combined dataset shape:", combined_df.shape)

print("\nCounts by source_dataset:")
print(combined_df["source_dataset"].value_counts())

print("\nCounts by class_label:")
print(combined_df["class_label"].value_counts())

print("\nColumns in combined dataset:")
print(combined_df.columns.tolist())

combined_csv_path = OUTPUT_DIR / "combined_metadata.csv"
combined_df.to_csv(combined_csv_path, index=False)

print(f"\nSaved combined metadata to: {combined_csv_path}")



Combined dataset shape: (12000, 4)

Counts by source_dataset:
source_dataset
ImageNet_1K_256       3000
MS_COCO_2017          3000
DiffusionDB           3000
SDXL_Generated_10K    3000
Name: count, dtype: int64

Counts by class_label:
class_label
real    6000
ai      6000
Name: count, dtype: int64

Columns in combined dataset:
['filename', 'source_dataset', 'class_label', 'image_path']

Saved combined metadata to: /content/combined_metadata.csv


In [6]:
# =========================
# Cell 4: Split combined metadata into exact train / validation / test sets
# =========================
TRAIN_COUNT_PER_SOURCE = 2100
VALIDATION_COUNT_PER_SOURCE = 450
TEST_COUNT_PER_SOURCE = 450

split_dfs = []

for dataset_name, group_df in combined_df.groupby("source_dataset"):
    # Shuffle each dataset independently
    group_df = group_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

    # Exact-count split
    train_df = group_df.iloc[:TRAIN_COUNT_PER_SOURCE].copy()
    validation_df = group_df.iloc[
        TRAIN_COUNT_PER_SOURCE:TRAIN_COUNT_PER_SOURCE + VALIDATION_COUNT_PER_SOURCE
    ].copy()
    test_df = group_df.iloc[
        TRAIN_COUNT_PER_SOURCE + VALIDATION_COUNT_PER_SOURCE:
        TRAIN_COUNT_PER_SOURCE + VALIDATION_COUNT_PER_SOURCE + TEST_COUNT_PER_SOURCE
    ].copy()

    train_df["subset"] = "train"
    validation_df["subset"] = "validation"
    test_df["subset"] = "test"

    split_dfs.extend([train_df, validation_df, test_df])

split_df = pd.concat(split_dfs, ignore_index=True)

train_metadata = split_df[split_df["subset"] == "train"].copy()
validation_metadata = split_df[split_df["subset"] == "validation"].copy()
test_metadata = split_df[split_df["subset"] == "test"].copy()

# Final shuffle of each subset independently
train_metadata = train_metadata.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
validation_metadata = validation_metadata.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
test_metadata = test_metadata.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

train_csv_path = OUTPUT_DIR / "train_metadata.csv"
validation_csv_path = OUTPUT_DIR / "validation_metadata.csv"
test_csv_path = OUTPUT_DIR / "test_metadata.csv"

train_metadata.to_csv(train_csv_path, index=False)
validation_metadata.to_csv(validation_csv_path, index=False)
test_metadata.to_csv(test_csv_path, index=False)

print("Train shape:", train_metadata.shape)
print("Validation shape:", validation_metadata.shape)
print("Test shape:", test_metadata.shape)

print("\nCounts by subset:")
print(pd.Series({
    "train": len(train_metadata),
    "validation": len(validation_metadata),
    "test": len(test_metadata),
}))

print("\nCounts by subset and source_dataset:")
print(split_df.groupby(["subset", "source_dataset"]).size())

print("\nCounts by subset and class_label:")
print(split_df.groupby(["subset", "class_label"]).size())

print(f"\nSaved train metadata to: {train_csv_path}")
print(f"Saved validation metadata to: {validation_csv_path}")
print(f"Saved test metadata to: {test_csv_path}")


Train shape: (8400, 5)
Validation shape: (1800, 5)
Test shape: (1800, 5)

Counts by subset:
train         8400
validation    1800
test          1800
dtype: int64

Counts by subset and source_dataset:
subset      source_dataset    
test        DiffusionDB            450
            ImageNet_1K_256        450
            MS_COCO_2017           450
            SDXL_Generated_10K     450
train       DiffusionDB           2100
            ImageNet_1K_256       2100
            MS_COCO_2017          2100
            SDXL_Generated_10K    2100
validation  DiffusionDB            450
            ImageNet_1K_256        450
            MS_COCO_2017           450
            SDXL_Generated_10K     450
dtype: int64

Counts by subset and class_label:
subset      class_label
test        ai              900
            real            900
train       ai             4200
            real           4200
validation  ai              900
            real            900
dtype: int64

Saved train metadata to

In [7]:
# =========================
# Cell 5: Optional validation of output CSV files
# =========================
for csv_name in [
    "combined_metadata.csv",
    "train_metadata.csv",
    "validation_metadata.csv",
    "test_metadata.csv",
]:
    csv_path = OUTPUT_DIR / csv_name
    if not csv_path.exists():
        print(f"Missing: {csv_path}")
    else:
        df = pd.read_csv(csv_path)
        print(f"{csv_name}: {df.shape}")


combined_metadata.csv: (12000, 4)
train_metadata.csv: (8400, 5)
validation_metadata.csv: (1800, 5)
test_metadata.csv: (1800, 5)
